In [1]:
from proteinage.utils import *
from proteinage.predict import ProtAge_predict
from proteinage.predict import ProtAge_predict_UKB
import random
import miceforest as mf
import pandas as pd
import datetime as dt

random.seed(7)

In [2]:
#data = pd.read_csv('../Data/Processed/Full/full_test.csv', index_col = 'eid')

In [3]:
#Import imputed Olink proteins
olink_imputed = pd.read_csv('../Data/Other_Biomarkers/proteins_set3_imputed_other_biomarkers.csv', index_col = 'eid')
olink_imputed.drop('age', axis=1, inplace=True) #Drop age column
olink_imputed.columns = olink_imputed.columns.str.upper()
olink_imputed.rename(columns={'C19ORF12': 'C19orf12', 'NTPROBNP': 'NTproBNP'}, inplace=True) #These two need to be different

In [4]:
#Use age per decimal
age_df = pd.read_csv('../Data/covar/UKB_Lifestyle.csv', index_col = 'eid')
age_df['age'] = age_df['age.0.0']
age_df = age_df[['age']]

#Left join with olink_imputed
olink_imputed = olink_imputed.join(age_df)
olink_imputed.columns

Index(['A1BG', 'AAMDC', 'AARSD1', 'ABCA2', 'ABHD14B', 'ABL1', 'ABO',
       'ABRAXAS2', 'ACAA1', 'ACADM',
       ...
       'ZHX2', 'ZNF174', 'ZNF75D', 'ZNF830', 'ZNRD2', 'ZNRF4', 'ZP3', 'ZP4',
       'ZPR1', 'age'],
      dtype='object', length=2921)

In [5]:
olink_imputed.columns

Index(['A1BG', 'AAMDC', 'AARSD1', 'ABCA2', 'ABHD14B', 'ABL1', 'ABO',
       'ABRAXAS2', 'ACAA1', 'ACADM',
       ...
       'ZHX2', 'ZNF174', 'ZNF75D', 'ZNF830', 'ZNRD2', 'ZNRF4', 'ZP3', 'ZP4',
       'ZPR1', 'age'],
      dtype='object', length=2921)

In [6]:
help(mf.ImputationKernel)

Help on class ImputationKernel in module miceforest.imputation_kernel:

class ImputationKernel(miceforest.imputed_data.ImputedData)
 |  ImputationKernel(data: pandas.core.frame.DataFrame, num_datasets: int = 1, variable_schema: Union[List[str], Dict[str, List[str]], NoneType] = None, imputation_order: Literal['ascending', 'descending', 'roman', 'latin'] = 'ascending', mean_match_candidates: Union[int, Dict[str, int]] = 5, mean_match_strategy: Union[str, Dict[str, str], NoneType] = 'normal', data_subset: Union[int, Dict[str, int]] = 0, initialize_empty: bool = False, save_all_iterations_data: bool = True, copy_data: bool = True, random_state: Union[int, numpy.random.mtrand.RandomState, NoneType] = None)
 |
 |  Creates a kernel dataset. This dataset can perform MICE on itself,
 |  and impute new data from models obtained during MICE.
 |
 |  Parameters
 |  ----------
 |  data : pandas.DataFrame.
 |      The data to be imputed.
 |  variable_schema : None or List[str] or Dict[str, str], def

In [7]:
# random_seed = 3456

# # Load protein data
# data_path = '../Data/Olink_UKBB_packageDataFreeze_19Jan2024/67864_olink_0.csv'
# data = pd.read_csv(data_path)
# data = data[data.columns[data.columns != 'X']]

# # Create a dictionary of variables use to impute
# exclude = ['eid', 'olink_batch', 'olink_plate', 'glipr1', 'npm1', 'pcolce']
# dont_impute = ['eid', 'olink_batch', 'olink_plate']
# column_dict = {col: [other_col for other_col in data.columns if other_col != col and other_col not in exclude] for col in data.columns if col not in dont_impute}

# # run miceforest imputation on multiple cores
# kds = mf.ImputationKernel(
#   data,
#   num_datasets=1,
#   variable_schema=column_dict,
#   random_state=random_seed
# )

# # run
# kds.mice(
#   iterations=5,
#   n_jobs=-1, 
#   verbose=True
# )

# # get the completed dataframe from the miceforest object
# olink_data_imputed = kds.complete_data()

In [8]:
# calculate ProtAge
model_file = 'ProteinAge_Model_Files/ProtAge_model.p'
#predictions = ProtAge_predict(data, model='ProtAge', model_file=model_file)

# calculate ProtAge20
model_file20 = 'ProteinAge_Model_Files/ProtAge20_model.p'
#predictions = ProtAge_predict(data, model='ProtAge20', model_file=model_file)

In [9]:
# calculate ProtAge and normalize proteomic data within-sample as we use UK Biobank data
predictions = ProtAge_predict_UKB(
    olink_imputed, 
    model='ProtAge', 
    model_path=model_file,
    # data should still be normalized. This replicates our published method
    normalize=True, 
    # should be age as a decimal using our published method
    age_col='age' 
)
# calculate ProtAge20 and normalize proteomic data within-sample as we use UK Biobank data
predictions20 = ProtAge_predict_UKB(
    olink_imputed, 
    model='ProtAge20', 
    model_path=model_file20,
    # data should still be normalized. This replicates our published method
    normalize=True, 
    # should be age as a decimal using our published method
    age_col='age' 
)

Normalizing protein values... 

/home/l.m.kuiper/miniconda3/envs/olink_aging_lieke/lib/python3.12/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


complete
ProtAge predictions complete
Normalizing protein values... 

/home/l.m.kuiper/miniconda3/envs/olink_aging_lieke/lib/python3.12/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


complete
ProtAge20 predictions complete


In [10]:
help(ProtAge_predict_UKB)

Help on function ProtAge_predict_UKB in module proteinage.predict:

ProtAge_predict_UKB(df, model='ProtAge', model_path=<class 'str'>, age_col=None, normalize=True)
    # Function to make predictions in UKB data only



In [15]:
(predictions20['predicted_scores'] == predictions20['predicted_values']).all() #Completely the same

True

In [16]:
(predictions['predicted_scores'] == predictions['predicted_values']).all() #Completely the same

True

In [17]:
proteinage = pd.DataFrame({'ProteinAge':predictions['predicted_scores'], 'ProteinAge20': predictions20['predicted_scores']}, index = olink_imputed.index)

In [18]:
proteinage

,ProteinAge,ProteinAge20
eid,,
1001997,59.625420,61.455433
1002479,55.384476,53.242527
1021769,49.882408,49.609694
1062421,50.482902,50.083580
1065061,62.625578,61.052725
...,...,...
6022822,63.921676,63.664471
6023322,47.060727,49.147120
6024184,49.666831,48.266241


In [19]:
proteinage.to_csv('../Data/Other_Biomarkers/ProteinAge.csv')

In [20]:
pwd()

'/home/projects/olink-aging/olink-aging/Other_Markers'